# One-at-a-Time (OAT) Manning's n Sensitivity Analysis

This notebook performs a **one-at-a-time (OAT)** sensitivity analysis on Manning's n values
using a 2D HEC-RAS model. Each land cover class is varied individually while holding all
others at their baseline values, producing sensitivity curves and a tornado diagram.

**Key Concepts:**
- OAT sensitivity: vary one parameter at a time to isolate its effect
- Manning's n range tables with configurable interval stepping
- Canonical roughness propagation: cloned geometry, plaintext `LCMann Table` edits, and fresh geometry preprocessing
- Distributed plan execution with RasRemote workers, falling back to local `RasCmdr.compute_parallel()`
- Sensitivity curves and tornado diagram visualization
- Spatial/global delta metrics for max WSE, inundation volume, face velocity, and arrival time

**Version note:** the Sabinal ScienceBase validation run exposed a HEC-RAS 6.6 DSS finalization
crash (`RasUnsteady.exe ... DSS 91 DSS.for`) after hydraulics completed. HEC-RAS 7.0 completed the
same p10/p14/p20/p54/p59/p65 plans cleanly with full final HDFs, so this notebook defaults to 7.0.

**Prerequisites:** HEC-RAS 7.0+ installed, ras-commander package, optional RasRemote worker JSON.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from ras_commander import (
        init_ras_project, RasCmdr, RasPlan, RasGeo, RasExamples,
        RasMap, RasMonteCarlo, RasCurrency
    )
    from ras_commander.hdf import HdfMesh, HdfResultsMesh, HdfResultsPlan
except ImportError:
    current_file = Path("__file__").resolve()
    parent_directory = current_file.parent.parent
    sys.path.append(str(parent_directory))
    from ras_commander import (
        init_ras_project, RasCmdr, RasPlan, RasGeo, RasExamples,
        RasMap, RasMonteCarlo, RasCurrency
    )
    from ras_commander.hdf import HdfMesh, HdfResultsMesh, HdfResultsPlan

## Configuration

All parameters for the analysis are defined here. Adjust `INTERVAL` to control
the granularity of the sensitivity sweep (smaller interval = more plans = longer runtime).

In [ ]:
# === PROJECT CONFIGURATION ===
PROJECT_NAME = os.getenv("RAS_COMMANDER_711_PROJECT_NAME", "Muncie")  # RasExamples project name
PROJECT_PATH_OVERRIDE = os.getenv("RAS_COMMANDER_711_PROJECT_PATH")    # Optional external RAS project folder
RAS_VERSION = os.getenv("RAS_COMMANDER_711_RAS_VERSION", "7.0")        # HEC-RAS version string
TEMPLATE_PLAN = os.getenv("RAS_COMMANDER_711_TEMPLATE_PLAN", "04")    # Plan number with 2D geometry

# === SENSITIVITY CONFIGURATION ===
INTERVAL = float(os.getenv("RAS_COMMANDER_711_INTERVAL", "0.02"))      # Manning's n step size for sweep

# === POINT OF INTEREST ===
# Defaults are for the Muncie example project (State Plane Indiana East, ft).
# For external projects, override with RAS_COMMANDER_711_POI_X/Y/LABEL.
POI_X = float(os.getenv("RAS_COMMANDER_711_POI_X", "408350.0"))
POI_Y = float(os.getenv("RAS_COMMANDER_711_POI_Y", "1802550.0"))
POI_LABEL = os.getenv("RAS_COMMANDER_711_POI_LABEL", "Mid-Floodplain POI")

# === EXECUTION ===
MAX_WORKERS = int(os.getenv("RAS_COMMANDER_711_MAX_WORKERS", "4"))     # Local parallel workers
NUM_CORES = int(os.getenv("RAS_COMMANDER_711_NUM_CORES", "2"))         # CPU cores per HEC-RAS instance

# === RASREMOTE EXECUTION ===
def _env_bool(name, default):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}

USE_REMOTE_WORKERS = _env_bool("RAS_COMMANDER_711_USE_REMOTE", True)
REQUESTED_REMOTE_WORKERS = tuple(
    label.strip()
    for label in os.getenv("RAS_COMMANDER_711_REMOTE_WORKERS", "CLB02,CLB04,CLB06,CLB10").split(",")
    if label.strip()
)
CLEAR_GEOMPRE = _env_bool(
    "RAS_COMMANDER_711_CLEAR_GEOMPRE", True
)  # Regenerate geometry preprocessor files after Manning's n edits


## Configure RasRemote Workers

Load the CLB RasRemote fleet from the gitignored worker configuration. Passwords are never printed.


In [ ]:
import json

# RasRemote Worker Setup
# Loads a worker fleet from working/remote_workers_clb.json (gitignored, has real credentials).
# Falls back to examples/remote_workers_clb_template.json (committed, redacted) for reference.
# Requested labels are filtered so this notebook cannot spill onto older CLB workers.
_cwd = Path.cwd()
if (_cwd / "remote_workers_clb_template.json").exists():
    _repo_root = _cwd.parent
    _examples_dir = _cwd
else:
    _repo_root = _cwd
    _examples_dir = _cwd / "examples"

_workers_json_real = _repo_root / "working" / "remote_workers_clb.json"
_workers_json_template = _examples_dir / "remote_workers_clb_template.json"
_requested_labels = {label.upper() for label in REQUESTED_REMOTE_WORKERS}

def _worker_label_from_name(name: str):
    normalized = name.upper().replace("CLB-", "CLB")
    for label in sorted(_requested_labels):
        if label in normalized:
            return label
    return None

remote_workers = []
remote_worker_status = pd.DataFrame()

if USE_REMOTE_WORKERS:
    try:
        from ras_commander.remote import load_workers_from_json

        _cfg_path = _workers_json_real if _workers_json_real.exists() else _workers_json_template
        _cfg_raw = json.loads(_cfg_path.read_text())

        _status_rows = []
        _enabled_requested_labels = set()
        for _entry in _cfg_raw.get("workers", []):
            _label = _worker_label_from_name(_entry.get("name", ""))
            if _label is None:
                continue
            if _entry.get("enabled", False):
                _enabled_requested_labels.add(_label)
            _status_rows.append({
                "label": _label,
                "name": _entry.get("name", ""),
                "enabled": bool(_entry.get("enabled", False)),
                "hostname": _entry.get("hostname", ""),
                "share_path": _entry.get("share_path", ""),
                "session_id": _entry.get("session_id", ""),
                "slots": (_entry.get("cores_total", 0) or 0) // max((_entry.get("cores_per_plan", 1) or 1), 1),
                "note": _entry.get("_note", ""),
            })

        _loaded_workers = load_workers_from_json(_cfg_path, enabled_only=True)
        remote_workers = [
            _worker for _worker in _loaded_workers
            if getattr(_worker, "worker_type", "") != "local"
            and _worker_label_from_name(getattr(_worker, "worker_id", "")) in _requested_labels
        ]

        _loaded_labels = {
            _worker_label_from_name(getattr(_worker, "worker_id", ""))
            for _worker in remote_workers
        }
        _loaded_labels.discard(None)
        _missing_labels = sorted(_requested_labels - _loaded_labels)
        _total_slots = sum(getattr(_worker, "max_parallel_plans", 1) or 1 for _worker in remote_workers)

        remote_worker_status = pd.DataFrame(_status_rows).sort_values(["label", "name"])
        print(remote_worker_status.to_string(index=False))
        print(f"Loaded {len(remote_workers)} RasRemote workers ({_total_slots} concurrent slots).")
        if _missing_labels:
            print(f"Requested worker labels not loaded: {', '.join(_missing_labels)}")
    except Exception as _e:
        print(f"Warning: could not load RasRemote workers ({_e}) -- falling back to local execution.")
        USE_REMOTE_WORKERS = False
        remote_workers = []
else:
    print("USE_REMOTE_WORKERS = False -- using local compute_parallel.")
    print(f"Worker config available at: {_workers_json_real}")


## Extract Example Project and Initialize

In [ ]:
if PROJECT_PATH_OVERRIDE:
    project_folder = Path(PROJECT_PATH_OVERRIDE).expanduser()
    project_source = "external override"
else:
    project_folder = RasExamples.extract_project(PROJECT_NAME)
    project_source = "RasExamples"

ras = init_ras_project(project_folder, RAS_VERSION)

print(f"Project: {project_folder}")
print(f"Source: {project_source}")
print(f"RAS version: {RAS_VERSION}")
print(f"Plans: {len(ras.plan_df)}")
print(f"Template plan: {TEMPLATE_PLAN}")
print()
print("Plan DataFrame:")
ras.plan_df[["plan_number", "Plan Title", "Geom File"]].head(10)


## Read Baseline Manning’s n Values

Extract the current Manning’s n values from the template geometry file.
These serve as the baseline for the OAT sensitivity analysis.

In [ ]:
# Get geometry file path from template plan
template_row = ras.plan_df[ras.plan_df["plan_number"] == TEMPLATE_PLAN].iloc[0]
template_geom_number = str(template_row["geometry_number"]).zfill(2)
geom_file = Path(template_row["Geom Path"])
if not geom_file.exists():
    geom_file = project_folder / f"{ras.project_name}.g{template_row['geometry_number']}"

# Read base overrides (primary Manning's n table)
base_overrides_df = RasGeo.get_mannings_baseoverrides(geom_file).copy()
base_overrides_df["Base Mannings n Value"] = pd.to_numeric(
    base_overrides_df["Base Mannings n Value"], errors="coerce"
)
base_overrides = dict(zip(
    base_overrides_df["Land Cover Name"],
    base_overrides_df["Base Mannings n Value"],
))

print("Base Manning's n Overrides:")
print(f"  Land covers: {len(base_overrides)}")
for name, n_val in base_overrides.items():
    print(f"  {name}: n = {n_val}")

# Read regional overrides
regional_overrides_df = RasGeo.get_mannings_regionoverrides(geom_file)
print()
print(f"Regional Overrides: {len(regional_overrides_df)} rows")
if not regional_overrides_df.empty:
    print(regional_overrides_df.to_string(index=False))


## Build Manning’s n Range Table

Define min/max bounds for each land cover class. Obstruction values (n >= 1.0,
typically buildings at n=100) are excluded from the sensitivity analysis since
they represent physical barriers, not roughness.

In [ ]:
# Standard Manning's n ranges by land cover type
# Source: HEC-RAS Reference Manual, Chow (1959)
MANNINGS_RANGES = {
    "Pasture":           (0.025, 0.050),
    "Brush":             (0.040, 0.120),
    "Trees":             (0.060, 0.200),
    "Low Intensity":     (0.040, 0.120),
    "High Intensity":    (0.060, 0.200),
    "Water":             (0.025, 0.050),
    "Cropland":          (0.020, 0.060),
    "Wetland":           (0.030, 0.100),
    "Barren":            (0.010, 0.035),
}

OBSTRUCTION_THRESHOLD = 1.0  # n values >= this are obstructions

# Build range table from base overrides
range_table = {}
excluded = {}

for name, baseline_n in base_overrides.items():
    if baseline_n >= OBSTRUCTION_THRESHOLD:
        excluded[name] = baseline_n
        continue
    
    if name in MANNINGS_RANGES:
        n_min, n_max = MANNINGS_RANGES[name]
    else:
        # Default: +/- 50% of baseline, clamped to [0.01, 0.5]
        n_min = max(0.01, baseline_n * 0.5)
        n_max = min(0.50, baseline_n * 1.5)
    
    range_table[name] = {
        "baseline": baseline_n,
        "min": n_min,
        "max": n_max,
        "steps": np.arange(n_min, n_max + INTERVAL / 2, INTERVAL)
    }

print("=== Sensitivity Range Table ===")
print(f"Variable land covers: {len(range_table)}")
print(f"Excluded (obstructions): {len(excluded)}")
print()

total_plans = sum(len(v["steps"]) for v in range_table.values())
print(f"Total plans to create: {total_plans}")
print()

for name, info in range_table.items():
    print(f'  {name}: baseline={info["baseline"]:.3f}, '
          f'range=[{info["min"]:.3f}, {info["max"]:.3f}], '
          f'steps={len(info["steps"])}')

if excluded:
    print(f"\nExcluded obstructions:")
    for name, n_val in excluded.items():
        print(f"  {name}: n = {n_val} (held constant)")

## Create OAT Sensitivity Plans

For each land cover, create a set of plans that vary only that land cover's
Manning's n while keeping all others at baseline. Each plan gets a cloned
geometry file, and the cloned plan is modified through
`RasMonteCarlo.make_mannings_apply_fn(path="plaintext")`, which edits the
plain-text `LCMann Table` in that clone's `.g##`. Execution must use
`clear_geompre=True` so HEC-RAS regenerates cached per-cell Manning's n.


In [ ]:
plan_registry = []  # Track all created plans
plan_counter = 0

# Canonical 2D land-cover roughness propagation path:
#   cloned .g## per plan + plaintext LCMann Table edit + clear_geompre=True at execution.
# This mirrors RasMonteCarlo.run_ensemble(..., clone_geom=True, clear_geompre=True)
# without building a stochastic ensemble.
mannings_apply_fn = RasMonteCarlo.make_mannings_apply_fn(
    zone_column_map={str(name): str(name) for name in base_overrides_df["Land Cover Name"]},
    path="plaintext",
)

for lc_name, info in range_table.items():
    print(f"\nCreating plans for: {lc_name} ({len(info['steps'])} steps)")
    
    for n_value in info["steps"]:
        plan_counter += 1
        plan_title = f"OAT_{str(lc_name)[:8]}_{n_value:.3f}"
        
        # Clone plan and geometry from template. Each sensitivity plan needs its own
        # geometry so its plaintext LCMann Table does not collide with other runs.
        new_plan_number = RasPlan.clone_plan(TEMPLATE_PLAN, ras_object=ras)
        new_geom_number = RasPlan.clone_geom(template_geom_number, ras_object=ras)
        
        # Link cloned plan to cloned geometry
        RasPlan.set_geom(new_plan_number, new_geom_number, ras_object=ras)
        
        # Set plan title
        plan_file = ras.project_folder / f"{ras.project_name}.p{new_plan_number}"
        RasPlan.set_plan_title(plan_file, plan_title)
        
        # Modify Manning's n through the hardened plaintext LCMann propagation path:
        # build a full row of baseline class n values, then perturb only the target class.
        sample_row = pd.Series({str(k): float(v) for k, v in base_overrides.items()})
        sample_row[str(lc_name)] = float(n_value)
        mannings_apply_fn(plan_file, sample_row, ras_object=ras)
        
        plan_registry.append({
            "land_cover": lc_name,
            "n_value": round(float(n_value), 4),
            "plan_number": new_plan_number,
            "geom_number": new_geom_number,
            "Plan Title": plan_title,
            "is_baseline": abs(float(n_value) - float(info["baseline"])) < 1e-6,
        })

# Refresh project state
ras = init_ras_project(project_folder, RAS_VERSION)

registry_df = pd.DataFrame(plan_registry)
print("\n=== Plan Registry ===")
print(f"Total plans created: {len(registry_df)}")
print(f"Land covers varied: {registry_df['land_cover'].nunique()}")
print("\nPlans per land cover:")
print(registry_df.groupby("land_cover").size())


## Execute All Plans

Run the sensitivity plans with RasRemote when configured, otherwise use local parallel execution.


In [ ]:
all_plan_numbers = registry_df["plan_number"].tolist()

if USE_REMOTE_WORKERS and remote_workers:
    from ras_commander.remote import compute_parallel_remote

    total_slots = sum(getattr(worker, "max_parallel_plans", 1) or 1 for worker in remote_workers)
    print(
        f"Executing {len(all_plan_numbers)} plans with RasRemote "
        f"on {len(remote_workers)} workers ({total_slots} slots)..."
    )

    remote_results = compute_parallel_remote(
        plan_numbers=all_plan_numbers,
        workers=remote_workers,
        ras_object=ras,
        num_cores=NUM_CORES,
        clear_geompre=CLEAR_GEOMPRE,
        force_rerun=True,
    )

    missing_remote = sorted(set(all_plan_numbers) - set(remote_results.keys()))
    if missing_remote:
        raise RuntimeError(f"RasRemote returned no result for plan(s): {missing_remote}")

    execution_summary = pd.DataFrame([
        {
            "plan_number": plan_num,
            "worker_id": result.worker_id,
            "success": result.success,
            "execution_time_sec": round(result.execution_time, 1),
            "hdf_path": result.hdf_path or "",
            "error_message": result.error_message or "",
        }
        for plan_num, result in sorted(remote_results.items())
    ])
    print(execution_summary.to_string(index=False))

    failed = execution_summary[~execution_summary["success"]]
    if not failed.empty:
        raise RuntimeError(f"{len(failed)} RasRemote plan(s) failed; see execution_summary above.")
else:
    print(f"Executing {len(all_plan_numbers)} plans locally with {MAX_WORKERS} workers...")
    local_result = RasCmdr.compute_parallel(
        plan_number=all_plan_numbers,
        max_workers=MAX_WORKERS,
        num_cores=NUM_CORES,
        clear_geompre=CLEAR_GEOMPRE,
        force_rerun=True,
        verify=True,
        ras_object=ras,
    )
    execution_summary = local_result.results_df.copy()
    failed_local = sorted(
        plan_num for plan_num in all_plan_numbers
        if not bool(local_result.execution_results.get(plan_num, False))
    )
    if failed_local:
        raise RuntimeError(f"{len(failed_local)} local plan(s) failed verification: {failed_local}")

# Refresh after execution and explicitly check the final HDFs. This catches the
# Sabinal failure mode where hydraulics finished but HEC-RAS 6.6 crashed while
# writing DSS output, leaving a tiny final HDF plus a full .tmp.hdf.
ras = init_ras_project(project_folder, RAS_VERSION)
post_run_rows = []
for plan_num in all_plan_numbers:
    hdf_path = RasCurrency.get_plan_hdf_path(plan_num, ras)
    hdf_path = Path(hdf_path) if hdf_path is not None else None
    exists = bool(hdf_path and hdf_path.exists())
    complete = bool(exists and RasCurrency.check_plan_hdf_complete(hdf_path))
    message_flags = {
        "writing_results_to_dss": False,
        "dss_for": False,
        "error_with_program": False,
        "finished_unsteady": False,
        "complete_process": False,
    }
    if exists:
        try:
            msgs = HdfResultsPlan.get_compute_messages_hdf_only(hdf_path)
            message_flags = {
                "writing_results_to_dss": "Writing Results to DSS" in msgs,
                "dss_for": "DSS.for" in msgs,
                "error_with_program": "Error with program" in msgs,
                "finished_unsteady": "Finished Unsteady Flow Simulation" in msgs,
                "complete_process": "Complete Process" in msgs,
            }
        except Exception as exc:
            message_flags["message_read_error"] = str(exc)

    post_run_rows.append({
        "plan_number": plan_num,
        "hdf_path": str(hdf_path) if hdf_path else "",
        "hdf_exists": exists,
        "hdf_complete": complete,
        "hdf_size_bytes": hdf_path.stat().st_size if exists else 0,
        **message_flags,
    })

post_run_qaqc_df = pd.DataFrame(post_run_rows)
display(post_run_qaqc_df)

bad_post_run = post_run_qaqc_df[
    (~post_run_qaqc_df["hdf_complete"])
    | post_run_qaqc_df["dss_for"]
    | post_run_qaqc_df["error_with_program"]
]
if not bad_post_run.empty:
    display(bad_post_run)
    raise RuntimeError(
        f"{len(bad_post_run)} plan(s) failed post-run HDF/DSS QAQC. "
        "Use HEC-RAS 7.0+ for Sabinal-style DSS finalization failures."
    )

print("Execution complete; all final HDFs passed post-run QAQC.")


## Verify Roughness Propagation

Before interpreting WSE differences, verify that at least one OAT roughness change reached the
preprocessed 2D geometry. This checks the cached `Cells Center Manning's n` dataset in the
cloned geometry HDFs. If this fails, the runs are hydraulically inert regardless of HEC-RAS
completion status. Converted legacy projects may still fail here if the land-cover association
cannot be sampled into 2D cells.


In [ ]:
import h5py


def _cell_mannings_n_from_geom_hdf(geom_hdf_path):
    geom_hdf_path = Path(geom_hdf_path)
    if not geom_hdf_path.exists():
        return None
    with h5py.File(geom_hdf_path, "r") as hdf_file:
        areas = hdf_file.get("Geometry/2D Flow Areas", {})
        for area_name in areas.keys():
            ds_path = f"Geometry/2D Flow Areas/{area_name}/Cells Center Manning's n"
            if ds_path in hdf_file:
                return np.asarray(hdf_file[ds_path][...], dtype=float)
    return None


propagation_rows = []
for lc_name, group in registry_df.groupby("land_cover"):
    if len(group) < 2:
        continue
    low_row = group.sort_values("n_value").iloc[0]
    high_row = group.sort_values("n_value").iloc[-1]
    low_hdf = RasCurrency.get_geom_hdf_path(low_row["plan_number"], ras)
    high_hdf = RasCurrency.get_geom_hdf_path(high_row["plan_number"], ras)
    if low_hdf is None or high_hdf is None:
        propagation_rows.append({
            "land_cover": lc_name,
            "low_geom": "",
            "high_geom": "",
            "status": "missing_geom_hdf_path",
            "max_delta_n": np.nan,
        })
        continue

    low_hdf = Path(low_hdf)
    high_hdf = Path(high_hdf)
    low_n = _cell_mannings_n_from_geom_hdf(low_hdf)
    high_n = _cell_mannings_n_from_geom_hdf(high_hdf)
    if low_n is None or high_n is None:
        propagation_rows.append({
            "land_cover": lc_name,
            "low_geom": low_hdf.name,
            "high_geom": high_hdf.name,
            "status": "missing_cell_mannings_n",
            "max_delta_n": np.nan,
        })
        continue
    if low_n.shape != high_n.shape:
        propagation_rows.append({
            "land_cover": lc_name,
            "low_geom": low_hdf.name,
            "high_geom": high_hdf.name,
            "status": "shape_mismatch",
            "max_delta_n": np.nan,
        })
        continue
    delta = np.abs(high_n - low_n)
    propagation_rows.append({
        "land_cover": lc_name,
        "low_geom": low_hdf.name,
        "high_geom": high_hdf.name,
        "low_n": float(low_row["n_value"]),
        "high_n": float(high_row["n_value"]),
        "status": "ok",
        "max_delta_n": float(np.nanmax(delta)),
        "changed_cells": int(np.count_nonzero(delta > 1e-8)),
    })

propagation_df = pd.DataFrame(propagation_rows)
display(propagation_df.sort_values("max_delta_n", ascending=False, na_position="last"))

valid_delta = propagation_df.loc[propagation_df["status"] == "ok", "max_delta_n"].dropna()
if valid_delta.empty or float(valid_delta.max()) <= 1e-8:
    raise RuntimeError(
        "Manning's n perturbations did not reach preprocessed cell n. "
        "Verify cloned geometries, plaintext LCMann edits, and clear_geompre=True."
    )

print(f"Roughness propagation confirmed: max |delta n| = {float(valid_delta.max()):.6g}")


## Extract Results at Point of Interest

For each executed plan, extract the maximum water surface elevation (WSE)
at the point of interest. This allows us to measure the sensitivity of WSE
to each land cover’s Manning’s n value.

In [ ]:
from scipy.spatial import cKDTree

results = []
extraction_failures = []

for _, row in registry_df.iterrows():
    plan_num = row["plan_number"]

    try:
        plan_row = ras.plan_df[ras.plan_df["plan_number"] == plan_num]
        if plan_row.empty:
            extraction_failures.append({
                "plan_number": plan_num,
                "land_cover": row["land_cover"],
                "n_value": row["n_value"],
                "reason": "not_found_in_plan_df",
            })
            continue

        hdf_path = plan_row["HDF_Results_Path"].iloc[0]
        if pd.isna(hdf_path) or not Path(hdf_path).exists():
            extraction_failures.append({
                "plan_number": plan_num,
                "land_cover": row["land_cover"],
                "n_value": row["n_value"],
                "reason": "missing_hdf_results",
            })
            continue

        hdf_path = Path(hdf_path)

        # Use the summary output: maximum water surface per mesh cell.
        max_ws = HdfResultsMesh.get_mesh_max_ws(hdf_path)
        if max_ws.empty:
            extraction_failures.append({
                "plan_number": plan_num,
                "land_cover": row["land_cover"],
                "n_value": row["n_value"],
                "reason": "empty_max_wse_summary",
            })
            continue

        coords = np.column_stack([max_ws.geometry.x, max_ws.geometry.y])
        tree = cKDTree(coords)
        dist, idx = tree.query([POI_X, POI_Y])
        nearest = max_ws.iloc[idx]
        max_wse = nearest["maximum_water_surface"]

        results.append({
            "land_cover": row["land_cover"],
            "n_value": row["n_value"],
            "plan_number": plan_num,
            "max_wse_ft": float(max_wse),
            "is_baseline": row["is_baseline"],
            "poi_cell_dist_ft": float(dist),
            "mesh_name": nearest.get("mesh_name", ""),
            "cell_id": int(nearest.get("cell_id", -1)),
        })

    except Exception as e:
        extraction_failures.append({
            "plan_number": plan_num,
            "land_cover": row["land_cover"],
            "n_value": row["n_value"],
            "reason": str(e),
        })

results_df = pd.DataFrame(results)
extraction_failures_df = pd.DataFrame(extraction_failures)
print()
print(f"Results extracted: {len(results_df)} of {len(registry_df)} plans")
if not extraction_failures_df.empty:
    display(extraction_failures_df)
    raise RuntimeError(
        f"WSE extraction failed for {len(extraction_failures_df)} registered plan(s)."
    )
if results_df.empty:
    raise RuntimeError("No WSE results were extracted from completed HDF files.")
print(f"POI cell distance: {results_df['poi_cell_dist_ft'].iloc[0]:.1f} ft")
print()
print(f"WSE range: {results_df['max_wse_ft'].min():.2f} - {results_df['max_wse_ft'].max():.2f} ft")
results_df.head(10)


## Sensitivity Curves

Plot Manning’s n vs. Max WSE for each land cover class. Each subplot shows
how varying one land cover’s roughness (while holding others constant)
affects the water surface elevation at the POI.

In [ ]:
land_covers = results_df["land_cover"].unique()
n_lc = len(land_covers)
ncols = min(3, n_lc)
nrows = int(np.ceil(n_lc / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
fig.suptitle(f"OAT Manning's n Sensitivity at {POI_LABEL}", fontsize=14, fontweight="bold")

for i, lc_name in enumerate(land_covers):
    ax = axes[i // ncols, i % ncols]
    lc_data = results_df[results_df["land_cover"] == lc_name].sort_values("n_value")
    
    ax.plot(lc_data["n_value"], lc_data["max_wse_ft"], "o-", color="steelblue", linewidth=2)
    
    # Mark baseline
    baseline_row = lc_data[lc_data["is_baseline"]]
    if not baseline_row.empty:
        ax.axvline(baseline_row["n_value"].iloc[0], color="red", linestyle="--", alpha=0.7, label="Baseline")
    
    ax.set_title(lc_name, fontsize=11)
    ax.set_xlabel("Manning's n")
    ax.set_ylabel("Max WSE (ft)")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

# Hide unused subplots
for j in range(n_lc, nrows * ncols):
    axes[j // ncols, j % ncols].set_visible(False)

plt.tight_layout()
plt.show()

## Tornado Diagram

The tornado diagram ranks land cover classes by their influence on WSE.
Wider bars indicate greater sensitivity — these are the parameters that
matter most for model calibration.

In [ ]:
# Calculate WSE range (max - min) for each land cover
tornado_data = []
for lc_name in land_covers:
    lc_data = results_df[results_df["land_cover"] == lc_name]
    baseline_wse = lc_data[lc_data["is_baseline"]]["max_wse_ft"]
    baseline_val = baseline_wse.iloc[0] if not baseline_wse.empty else lc_data["max_wse_ft"].median()
    
    tornado_data.append({
        "land_cover": lc_name,
        "wse_min": lc_data["max_wse_ft"].min(),
        "wse_max": lc_data["max_wse_ft"].max(),
        "wse_range": lc_data["max_wse_ft"].max() - lc_data["max_wse_ft"].min(),
        "wse_baseline": baseline_val,
        "n_min": lc_data["n_value"].min(),
        "n_max": lc_data["n_value"].max(),
        "n_baseline": range_table[lc_name]["baseline"]
    })

tornado_df = pd.DataFrame(tornado_data).sort_values("wse_range", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, len(tornado_df) * 0.8)))

baseline_global = tornado_df["wse_baseline"].median()
y_pos = range(len(tornado_df))

for i, (_, row) in enumerate(tornado_df.iterrows()):
    left = row["wse_min"] - row["wse_baseline"]
    right = row["wse_max"] - row["wse_baseline"]
    
    ax.barh(i, right, left=0, height=0.6, color="steelblue", alpha=0.8)
    ax.barh(i, left, left=0, height=0.6, color="coral", alpha=0.8)
    
    ax.text(right + 0.01, i, f"+{right:.2f}", va="center", fontsize=9)
    ax.text(left - 0.01, i, f"{left:.2f}", va="center", ha="right", fontsize=9)

ax.set_yticks(list(y_pos))
ax.set_yticklabels([f'{r["land_cover"]} (n={r["n_baseline"]:.3f})' for _, r in tornado_df.iterrows()])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Change in Max WSE from Baseline (ft)")
ax.set_title(f"Manning's n Sensitivity Tornado Diagram\n{POI_LABEL}", fontweight="bold")
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Sensitivity Ranking (most to least influential) ===")
for _, row in tornado_df.iloc[::-1].iterrows():
    print(f'  {row["land_cover"]}: WSE range = {row["wse_range"]:.3f} ft '
          f'(n: {row["n_min"]:.3f} to {row["n_max"]:.3f})')

## Spatial and Global Delta Metrics

A single POI can miss hydraulic response elsewhere in the mesh. This section compares each land-cover sweep across the full 2D area and writes reviewable spatial outputs:

- cell-centered max-WSE range and high-minus-low delta grids
- mesh-wide max-inundation volume and wetted-area summaries
- face max-velocity range and high-minus-low delta grids
- first-wetting arrival-time range using a configurable depth threshold

The class-envelope metrics compare the tested min-to-max Manning's n range for each land-cover class, so they do not depend on a baseline plan being available.

In [ ]:
from ras_commander.hdf.HdfBase import HdfBase

SPATIAL_DELTA_DEPTH_THRESHOLD = float(
    os.getenv("RAS_COMMANDER_711_ARRIVAL_DEPTH_THRESHOLD", "0.10")
)
SPATIAL_DELTA_MESH = str(results_df["mesh_name"].dropna().iloc[0]) if "mesh_name" in results_df else "2D grids"
spatial_delta_dir = project_folder / "oat_spatial_global_delta"
spatial_delta_dir.mkdir(exist_ok=True)

ACRE_FT = 43560.0
ACRE = 43560.0


def _required_hdf_dataset(hdf_file, hdf_path):
    if hdf_path not in hdf_file:
        raise KeyError(f"Missing HDF dataset: {hdf_path}")
    return hdf_file[hdf_path]


def _volume_at_wse(max_wse, volume_info, volume_values):
    volumes = np.zeros(len(max_wse), dtype=float)
    for cell_id, (start, count) in enumerate(volume_info[:, :2].astype(int)):
        if count <= 0 or not np.isfinite(max_wse[cell_id]):
            volumes[cell_id] = np.nan
            continue
        table = volume_values[start : start + count]
        elevations = np.asarray(table[:, 0], dtype=float)
        cell_volumes = np.asarray(table[:, 1], dtype=float)
        volumes[cell_id] = np.interp(
            float(max_wse[cell_id]),
            elevations,
            cell_volumes,
            left=0.0,
            right=float(cell_volumes[-1]),
        )
    return volumes


def _arrival_hours(water_surface_ts, cell_min_elev, time_hours, depth_threshold):
    depth_ts = water_surface_ts - cell_min_elev.reshape(1, -1)
    wet = depth_ts > depth_threshold
    any_wet = wet.any(axis=0)
    first_idx = np.argmax(wet, axis=0)
    arrival = np.full(wet.shape[1], np.nan, dtype=float)
    arrival[any_wet] = time_hours[first_idx[any_wet]]
    return arrival


def _read_spatial_delta_arrays(hdf_path, mesh_name):
    hdf_path = Path(hdf_path)
    with h5py.File(hdf_path, "r") as hdf_file:
        summary_base = (
            "Results/Unsteady/Output/Output Blocks/Base Output/"
            f"Summary Output/2D Flow Areas/{mesh_name}"
        )
        ts_base = (
            "Results/Unsteady/Output/Output Blocks/Base Output/"
            f"Unsteady Time Series/2D Flow Areas/{mesh_name}"
        )
        geom_base = f"Geometry/2D Flow Areas/{mesh_name}"

        max_wse = np.asarray(
            _required_hdf_dataset(hdf_file, f"{summary_base}/Maximum Water Surface")[0, :],
            dtype=float,
        )
        max_face_velocity = np.asarray(
            _required_hdf_dataset(hdf_file, f"{summary_base}/Maximum Face Velocity")[0, :],
            dtype=float,
        )
        cell_min_elev = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/Cells Minimum Elevation")[()],
            dtype=float,
        )
        cell_area = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/Cells Surface Area")[()],
            dtype=float,
        )
        centers = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/Cells Center Coordinate")[()],
            dtype=float,
        )
        face_points = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/FacePoints Coordinate")[()],
            dtype=float,
        )
        face_point_indexes = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/Faces FacePoint Indexes")[()],
            dtype=int,
        )
        volume_info = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/Cells Volume Elevation Info")[()],
            dtype=int,
        )
        volume_values = np.asarray(
            _required_hdf_dataset(hdf_file, f"{geom_base}/Cells Volume Elevation Values")[()],
            dtype=float,
        )
        time_days = np.asarray(
            _required_hdf_dataset(
                hdf_file,
                "Results/Unsteady/Output/Output Blocks/Base Output/Unsteady Time Series/Time",
            )[()],
            dtype=float,
        )
        water_surface_ts = np.asarray(
            _required_hdf_dataset(hdf_file, f"{ts_base}/Water Surface")[()],
            dtype=float,
        )
        crs = HdfBase.get_projection(hdf_file)

    max_depth = np.maximum(max_wse - cell_min_elev, 0.0)
    return {
        "max_wse": max_wse,
        "max_depth": max_depth,
        "max_face_velocity": max_face_velocity,
        "cell_area": cell_area,
        "cell_volume": _volume_at_wse(max_wse, volume_info, volume_values),
        "arrival_hours": _arrival_hours(
            water_surface_ts,
            cell_min_elev,
            time_days * 24.0,
            SPATIAL_DELTA_DEPTH_THRESHOLD,
        ),
        "centers": centers,
        "face_points": face_points,
        "face_point_indexes": face_point_indexes,
        "crs": crs,
    }


def _finite_delta_stats(values, prefix):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return {
            f"{prefix}_abs_max": np.nan,
            f"{prefix}_abs_p95": np.nan,
            f"{prefix}_abs_mean": np.nan,
            f"{prefix}_count": 0,
            f"{prefix}_abs_ge_0_01": 0,
            f"{prefix}_abs_ge_0_10": 0,
        }
    abs_arr = np.abs(arr)
    return {
        f"{prefix}_abs_max": float(np.nanmax(abs_arr)),
        f"{prefix}_abs_p95": float(np.nanpercentile(abs_arr, 95)),
        f"{prefix}_abs_mean": float(np.nanmean(abs_arr)),
        f"{prefix}_count": int(arr.size),
        f"{prefix}_abs_ge_0_01": int(np.sum(abs_arr >= 0.01)),
        f"{prefix}_abs_ge_0_10": int(np.sum(abs_arr >= 0.10)),
    }


def _nan_range(stack):
    valid_count = np.isfinite(stack).sum(axis=0)
    result = np.full(stack.shape[1], np.nan, dtype=float)
    valid = valid_count >= 2
    if np.any(valid):
        result[valid] = np.nanmax(stack[:, valid], axis=0) - np.nanmin(stack[:, valid], axis=0)
    return result


spatial_arrays_by_plan = {}
spatial_plan_rows = []

for _, row in results_df.sort_values(["land_cover", "n_value"]).iterrows():
    plan_num = row["plan_number"]
    plan_row = ras.plan_df[ras.plan_df["plan_number"] == plan_num]
    if plan_row.empty:
        continue
    hdf_path = Path(plan_row["HDF_Results_Path"].iloc[0])
    arrays = _read_spatial_delta_arrays(hdf_path, SPATIAL_DELTA_MESH)
    spatial_arrays_by_plan[plan_num] = arrays
    wet = arrays["max_depth"] > SPATIAL_DELTA_DEPTH_THRESHOLD
    arrival = arrays["arrival_hours"]
    spatial_plan_rows.append({
        "plan_number": plan_num,
        "land_cover": row["land_cover"],
        "n_value": row["n_value"],
        "max_wse_global_ft": float(np.nanmax(arrays["max_wse"])),
        "max_depth_global_ft": float(np.nanmax(arrays["max_depth"])),
        "max_inundation_volume_acft": float(np.nansum(arrays["cell_volume"]) / ACRE_FT),
        "wet_area_ac_gt_threshold": float(np.nansum(arrays["cell_area"][wet]) / ACRE),
        "max_face_velocity_model_units_per_s": float(np.nanmax(np.abs(arrays["max_face_velocity"]))),
        "arrival_cell_count_gt_threshold": int(np.sum(np.isfinite(arrival))),
        "arrival_latest_hr": float(np.nanmax(arrival)) if np.isfinite(arrival).any() else np.nan,
    })

spatial_plan_metrics_df = pd.DataFrame(spatial_plan_rows)

spatial_class_rows = []
spatial_cell_grids = {}
spatial_face_grids = {}

for land_cover, lc_data in results_df.groupby("land_cover"):
    lc_data = lc_data.sort_values("n_value")
    plan_numbers = lc_data["plan_number"].tolist()
    arrays_list = [spatial_arrays_by_plan[p] for p in plan_numbers]

    wse_stack = np.stack([a["max_wse"] for a in arrays_list], axis=0)
    velocity_stack = np.stack([a["max_face_velocity"] for a in arrays_list], axis=0)
    arrival_stack = np.stack([a["arrival_hours"] for a in arrays_list], axis=0)
    volumes = np.array([np.nansum(a["cell_volume"]) / ACRE_FT for a in arrays_list], dtype=float)
    wet_areas = np.array([
        np.nansum(a["cell_area"][a["max_depth"] > SPATIAL_DELTA_DEPTH_THRESHOLD]) / ACRE
        for a in arrays_list
    ])

    wse_range = _nan_range(wse_stack)
    velocity_range = _nan_range(velocity_stack)
    arrival_range = _nan_range(arrival_stack)
    wse_high_minus_low = arrays_list[-1]["max_wse"] - arrays_list[0]["max_wse"]
    velocity_high_minus_low = arrays_list[-1]["max_face_velocity"] - arrays_list[0]["max_face_velocity"]
    arrival_high_minus_low = arrays_list[-1]["arrival_hours"] - arrays_list[0]["arrival_hours"]

    spatial_cell_grids[str(land_cover)] = {
        "wse_range": wse_range,
        "wse_high_minus_low": wse_high_minus_low,
        "arrival_range": arrival_range,
        "arrival_high_minus_low": arrival_high_minus_low,
    }
    spatial_face_grids[str(land_cover)] = {
        "velocity_range": velocity_range,
        "velocity_high_minus_low": velocity_high_minus_low,
    }

    summary_row = {
        "land_cover": land_cover,
        "plans": len(plan_numbers),
        "n_min": float(lc_data["n_value"].min()),
        "n_max": float(lc_data["n_value"].max()),
        "max_inundation_volume_range_acft": float(np.nanmax(volumes) - np.nanmin(volumes)),
        "max_inundation_volume_high_minus_low_acft": float(volumes[-1] - volumes[0]),
        "wet_area_range_ac_gt_threshold": float(np.nanmax(wet_areas) - np.nanmin(wet_areas)),
        "wet_area_high_minus_low_ac_gt_threshold": float(wet_areas[-1] - wet_areas[0]),
    }
    summary_row.update(_finite_delta_stats(wse_range, "max_wse_range_ft"))
    summary_row.update(_finite_delta_stats(wse_high_minus_low, "max_wse_high_minus_low_ft"))
    summary_row.update(_finite_delta_stats(velocity_range, "max_face_velocity_range_model_units_per_s"))
    summary_row.update(_finite_delta_stats(velocity_high_minus_low, "max_face_velocity_high_minus_low_model_units_per_s"))
    summary_row.update(_finite_delta_stats(arrival_range, "arrival_range_hr"))
    summary_row.update(_finite_delta_stats(arrival_high_minus_low, "arrival_high_minus_low_hr"))
    spatial_class_rows.append(summary_row)

spatial_class_delta_df = pd.DataFrame(spatial_class_rows).sort_values(
    "max_inundation_volume_range_acft",
    ascending=False,
)

spatial_plan_metrics_csv = spatial_delta_dir / "oat_spatial_plan_global_metrics.csv"
spatial_class_delta_csv = spatial_delta_dir / "oat_spatial_class_delta_summary.csv"
spatial_plan_metrics_df.to_csv(spatial_plan_metrics_csv, index=False)
spatial_class_delta_df.to_csv(spatial_class_delta_csv, index=False)

try:
    import geopandas as gpd
    from shapely.geometry import LineString, Point

    reference_arrays = next(iter(spatial_arrays_by_plan.values()))
    centers = reference_arrays["centers"]
    cell_df = pd.DataFrame({
        "mesh_name": SPATIAL_DELTA_MESH,
        "cell_id": np.arange(len(centers), dtype=int),
        "geometry": [Point(float(x), float(y)) for x, y in centers],
    })
    for land_cover, grids in spatial_cell_grids.items():
        safe_lc = str(land_cover).replace(" ", "_")
        cell_df[f"wse_rng_{safe_lc}"] = grids["wse_range"]
        cell_df[f"wse_hilo_{safe_lc}"] = grids["wse_high_minus_low"]
        cell_df[f"arr_rng_{safe_lc}"] = grids["arrival_range"]
        cell_df[f"arr_hilo_{safe_lc}"] = grids["arrival_high_minus_low"]
    cell_delta_gpkg = spatial_delta_dir / "oat_cell_wse_arrival_delta_grids.gpkg"
    gpd.GeoDataFrame(cell_df, geometry="geometry", crs=reference_arrays["crs"]).to_file(
        cell_delta_gpkg,
        layer="cell_deltas",
        driver="GPKG",
    )

    face_points = reference_arrays["face_points"]
    face_geometries = []
    for fp0, fp1 in reference_arrays["face_point_indexes"]:
        face_geometries.append(LineString([
            (float(face_points[fp0, 0]), float(face_points[fp0, 1])),
            (float(face_points[fp1, 0]), float(face_points[fp1, 1])),
        ]))
    face_df = pd.DataFrame({
        "mesh_name": SPATIAL_DELTA_MESH,
        "face_id": np.arange(len(face_geometries), dtype=int),
        "geometry": face_geometries,
    })
    for land_cover, grids in spatial_face_grids.items():
        safe_lc = str(land_cover).replace(" ", "_")
        face_df[f"vel_rng_{safe_lc}"] = grids["velocity_range"]
        face_df[f"vel_hilo_{safe_lc}"] = grids["velocity_high_minus_low"]
    face_delta_gpkg = spatial_delta_dir / "oat_face_velocity_delta_grids.gpkg"
    gpd.GeoDataFrame(face_df, geometry="geometry", crs=reference_arrays["crs"]).to_file(
        face_delta_gpkg,
        layer="face_deltas",
        driver="GPKG",
    )
    print(f"Cell delta grids: {cell_delta_gpkg}")
    print(f"Face velocity delta grids: {face_delta_gpkg}")
except Exception as exc:
    print(f"Spatial GeoPackage export skipped: {exc}")

print(f"Spatial/global plan metrics: {spatial_plan_metrics_csv}")
print(f"Spatial/global class summary: {spatial_class_delta_csv}")
display(spatial_class_delta_df[[
    "land_cover",
    "n_min",
    "n_max",
    "max_wse_range_ft_abs_max",
    "max_wse_range_ft_abs_p95",
    "max_inundation_volume_range_acft",
    "wet_area_range_ac_gt_threshold",
    "arrival_range_hr_abs_max",
    "arrival_range_hr_abs_p95",
    "max_face_velocity_range_model_units_per_s_abs_p95",
]])

## Export Results

Save the full results table and tornado summary as CSV files for
further analysis or reporting.

In [ ]:
# Full results table
results_csv = project_folder / "oat_sensitivity_results.csv"
results_df.to_csv(results_csv, index=False)
print(f"Full results: {results_csv}")

# Tornado summary
tornado_csv = project_folder / "oat_sensitivity_tornado.csv"
tornado_df.to_csv(tornado_csv, index=False)
print(f"Tornado summary: {tornado_csv}")

print(f"\nTotal plans analyzed: {len(results_df)}")
print(f"Land covers varied: {results_df['land_cover'].nunique()}")
print(f"Overall WSE range: {results_df['max_wse_ft'].max() - results_df['max_wse_ft'].min():.3f} ft")

## Cleanup

Remove the extracted example project to free disk space.

In [ ]:
import shutil

example_projects_dir = project_folder.parent
if example_projects_dir.name == "example_projects" and example_projects_dir.exists():
    shutil.rmtree(example_projects_dir, ignore_errors=True)
    print(f"Cleaned up: {example_projects_dir}")
else:
    print(f"Skipping cleanup: {example_projects_dir}")